In [ ]:
from pathlib import Path
import pathlib
temp = pathlib.PosixPath
pathlib.PosixPath = pathlib.WindowsPath

from fastai.vision.all import *
# from fastai.vision.all import load_learner
import numpy as np
from PIL import Image
from IPython.display import display
import cv2

np.int = np.int32 # Need to add this to make current model work

In [ ]:
# *** FUNCTIONS ***

# The colorblind friendly version
def add_mask2(source: Image.Image, truth: np.ndarray, pred: np.ndarray) -> Image.Image:
    """
    Given source black and white image, true and predicted mask this function returns image
    with areas colored as following:
    - Blue - annotation only
    - Green - prediction only
    - Purple - overlap
    """
    # Convert the source image to RGBA mode
    source = source.convert('RGBA')

    # Create an empty mask with an additional alpha channel
    M = np.zeros((*truth.shape, 4), dtype=np.uint8)

    # Set the blue channel for true annotations, orange channel for predictions, and alpha channel for visibility
    M[:, :, 2] = truth[:, :] * 255  # Blue for truth
    M[:, :, 1] = pred[:, :] * 128   # Green for prediction
    M[:, :, 0] = (truth & pred) * 255  # Purple for overlap
    M[:, :, 3] = ((truth > 0) | (pred > 0)) * 75  # Alpha for transparency (75/255)

    # Create an RGBA mask from the numpy array
    mask = Image.fromarray(M, 'RGBA')

    # Overlay the mask on the source image
    return Image.alpha_composite(source, mask)

def pixels2area(n: int) -> float:
    """Converts number of pixels into area in um^2"""
    return n * 0.084 * 0.084

def pixels2areaLOKI4(n: int) -> float:
    """Converts number of pixels into area in um^2"""
    return n * 0.092 * 0.092

# Calculate major and minor axes from mask # TODO: clean, initialize to NaN if no ellipse fit, seems like major and minor are flipped
def calculate_major_minor_axes(mask):
    """
    Calculate the longest major axis and minor axis of a binary mask.

    Args:
        mask (numpy.ndarray): A 2D binary NumPy array (0 for background, 1 for object).

    Returns:
        tuple: (major_axis_length, minor_axis_length)
    """
    if not isinstance(mask, np.ndarray):
        # raise ValueError("Input mask must be a NumPy array.")
        return np.nan, np.nan
        
    # Convert the binary mask (0s and 1s) to 8-bit format (0 and 255)
    binary_mask = (mask * 255).astype(np.uint8)

    # Find contours
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        # raise ValueError("No contours found in the mask.")
        return np.nan, np.nan

    # Get the largest contour (assuming the main object is the largest)
    largest_contour = max(contours, key=cv2.contourArea)

    # Fit an ellipse to the largest contour
    if len(largest_contour) >= 5:  # At least 5 points needed to fit an ellipse
        ellipse = cv2.fitEllipse(largest_contour)
        (_, axes, _) = ellipse  # axes contains (major_axis, minor_axis)
        
        # Ensure the major axis is always the longer one
        major_axis_length, minor_axis_length = max(axes), min(axes)
    else:
        # raise ValueError("Not enough points to fit an ellipse.")
        return np.nan, np.nan  # Return NaN if not enough points to fit an ellipse

    return major_axis_length, minor_axis_length


# Other modules needed from training
class CombinedLoss:
    "Dice and Focal combined"
    def __init__(self, axis=1, smooth=1., alpha=1.):
        store_attr()
        self.focal_loss = FocalLossFlat(axis=axis)
        self.dice_loss =  DiceLoss(axis, smooth)
        
    def __call__(self, pred, targ):
        return self.focal_loss(pred, targ) + self.alpha * self.dice_loss(pred, targ)
    
    def decodes(self, x):    return x.argmax(dim=self.axis)
    def activation(self, x): return F.softmax(x, dim=self.axis)


def IoU(preds:Tensor, targs:Tensor, eps:float=1e-8):
    """Computes the Jaccard loss, a.k.a the IoU loss.
    Notes: [Batch size,Num classes,Height,Width]
    Args:
        targs: a tensor of shape [B, H, W] or [B, 1, H, W].
        preds: a tensor of shape [B, C, H, W]. Corresponds to
            the raw output or logits of the model. (prediction)
        eps: added to the denominator for numerical stability.
    Returns:
        iou: the average class intersection over union value 
             for multi-class image segmentation
    """
    num_classes = preds.shape[1]
    
    # Single class segmentation?
    if num_classes == 1:
        true_1_hot = torch.eye(num_classes + 1)[targs.squeeze(1)]
        true_1_hot = true_1_hot.permute(0, 3, 1, 2).float()
        true_1_hot_f = true_1_hot[:, 0:1, :, :]
        true_1_hot_s = true_1_hot[:, 1:2, :, :]
        true_1_hot = torch.cat([true_1_hot_s, true_1_hot_f], dim=1)
        pos_prob = torch.sigmoid(preds)
        neg_prob = 1 - pos_prob
        probas = torch.cat([pos_prob, neg_prob], dim=1)
        
    # Multi-class segmentation
    else:
        # Convert target to one-hot encoding
        # true_1_hot = torch.eye(num_classes)[torch.squeeze(targs,1)]
        true_1_hot = torch.eye(num_classes)[targs.squeeze(1)]
        
        # Permute [B,H,W,C] to [B,C,H,W]
        true_1_hot = true_1_hot.permute(0, 3, 1, 2).float()
        
        # Take softmax along class dimension; all class probs add to 1 (per pixel)
        probas = F.softmax(preds, dim=1)
        
    true_1_hot = true_1_hot.type(preds.type())
    
    # Sum probabilities by class and across batch images
    dims = (0,) + tuple(range(2, targs.ndimension()))
    intersection = torch.sum(probas * true_1_hot, dims) # [class0,class1,class2,...]
    cardinality = torch.sum(probas + true_1_hot, dims)  # [class0,class1,class2,...]
    union = cardinality - intersection
    iou = (intersection / (union + eps)).mean()   # find mean of class IoU values
    return iou

def mask_outside_region(image_path, mask):
    image = cv2.imread(image_path)
    mask = (mask > 0.5).astype(np.uint8)  # Convert mask to binary
    masked_image = image.copy()
    masked_image[mask == 0] = 0  # Set all pixels outside the mask to black
    return masked_image
    
def area2mass(A: float) -> float:
    """Converts area in um^2 into mass in mg"""
    return 0.197 * (A ** 1.38)

def image2segmentation_path(imgpath: Path) -> Path:
    return Path(str(imgpath).replace("images_processed", "segmentations_processed"))

def segmentation2image_path(imgpath: Path) -> Path:
    return Path(str(imgpath).replace("segmentations_processed","images_processed"))


# For calculating the different model performance metrics
# Precision: TP/(TP+FP) (IoU)
# Recall/Sensitivity: TP/(TP+FN)
# F1 Score: (2xPrecisionxRecall)/(Precision+Recall)
# Specificity: TN/(TN+FP) (negative results/true negatives)... might not be useful for the blanks
def calculate_performance_metrics(mask1, mask2):
    # Ensure the masks have the same shape
    if mask1.shape != mask2.shape:
        raise ValueError("Masks must have the same shape.")
    
    # Calculate the intersection and area of annotated mask
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    mask1_area = np.sum(mask1)
    
    # Calculate the percentage overlap
    if mask1_area == 0:
        IoU = 0.0  # Avoid division by zero if both masks are empty
        Recall = 0.0 #  percent overlap of the prediction relative to the annotated mask
        F1score = 0.0
    else:
        IoU = (intersection / union) * 100
        Recall = (intersection / mask1_area) * 100
        F1Score = 2*(IoU*Recall)/(IoU+Recall)
        
    return IoU, Recall, F1Score


## Load 4 models in the pipeline
lipid classification --> prosome segmentation --> lipid segmentation/lipid segmentation with prosome crop

In [ ]:
modelLipidClassifier = load_learner("models/resnet34_lipid_classifier.pkl", cpu=True)  # File models/learner.pkl
modelLipidClassifier.load("resnet34_lipid_classifier")  

modelProsome = load_learner("models/model_resnet34_prosomeUVP6_deamundp1.pkl", cpu=True)  # File models/learner.pkl
modelProsome.load("model_resnet34_prosomeUVP6_deamundp1")  

modelLipid = load_learner("models/model_resnet34_lipidUVP6_deamundp1.pkl", cpu=True)  # File models/learner.pkl
modelLipid.load("model_resnet34_lipidUVP6_deamundp1")  

modelLipidPcrop = load_learner("models/model_resnet34_lipidcroppedUVP6_deamundp1.pkl", cpu=True)  # File models/learner.pkl
modelLipidPcrop.load("model_resnet34_lipidcroppedUVP6_deamundp1")  

# Prediction of new images UVP6
Pipeline: lipid classification --> prosome segmentation --> lipid segmentation/lipid segmentation with prosome crop


In [ ]:
imgs = sorted(list(Path("../uvp6_de_amund23p2_predictions/images_for_prediction").glob("*.png")))
# imgs = imgs[:10]

In [ ]:
# Processing and outputing the prediction metrics for PROSOME segmentation

# Initialize an empty list to store output data metrics
outdata = []

# Image directory
save_dir = "prediction_outputs/model_resnet34_prosomeUVP6_deamundp2"

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

for pimg in imgs:
    impath = pimg

    # Predict if lipid sac can be manually segmented
    predLipidSac, pred_idx, probs = modelLipidClassifier.predict(impath)
    prob_0, prob_1 = probs.tolist()

    # Predict prosome
    maskProsome, *_ = modelProsome.predict(impath)
    im_pred_mask_prosome = maskProsome.numpy()
    pixels_predicted_prosome = (im_pred_mask_prosome > 0.5).sum()
    prosome_major_predicted, prosome_minor_predicted = calculate_major_minor_axes(im_pred_mask_prosome)

    # Predict lipid
    maskLipid, *_ = modelLipid.predict(impath)
    im_pred_mask_lipid = maskLipid.numpy()
    pixels_predicted_lipid = (im_pred_mask_lipid > 0.5).sum()
    lipid_major_predicted, lipid_minor_predicted = calculate_major_minor_axes(im_pred_mask_lipid)

    # Crop the prosome out first then apply the lipid model to that
    masked_image = mask_outside_region(impath, im_pred_mask_prosome)
    cv2.imwrite("masked_temp.png", masked_image)
    
    # Predict lipid on the masked image
    maskLipidPcrop, *_ = modelLipidPcrop.predict("masked_temp.png")
    im_pred_mask_lipidPcrop = maskLipidPcrop.numpy()
    pixels_predicted_lipidPcrop = (im_pred_mask_lipidPcrop > 0.5).sum()
    lipidPcrop_major_predicted, lipidPcrop_minor_predicted = calculate_major_minor_axes(im_pred_mask_lipidPcrop)

    
    # Calculate lipid fullness as pixels lipid/pixels prosome
    lipidfullness1 = pixels_predicted_lipid / pixels_predicted_prosome * 100
    lipidfullness2 = pixels_predicted_lipidPcrop / pixels_predicted_prosome * 100
    
    # Print the images with predicted segments?? # TODO maybe loop it that this puts the lipid mask on the prosome mask
    im = Image.open(impath)
    # # A blank mask which is useful for displaying the image
    # im_truth = np.zeros_like(im_pred_mask_prosome)

    img_with_masks = add_mask2(im, im_pred_mask_lipidPcrop, im_pred_mask_prosome)
    # Blue - annotation
    # Green - prediction
    # Purple - overlap
    image_rgb = img_with_masks.convert("RGB")


    out_dir, out_fname = os.path.split(pimg) # Separate the directory from the file name
    filename_without_ext, ext = os.path.splitext(out_fname) # separate the extention
    updated_filename = f"{filename_without_ext}_overlapRGB{ext}"

    # Define the output path
    output_path = os.path.join(save_dir, updated_filename)
    image_rgb.save(output_path)
    
    
    outdata.append([out_dir, out_fname, 
                    predLipidSac,prob_0,prob_1,
                    pixels_predicted_prosome, 
                    pixels_predicted_lipid, pixels_predicted_lipidPcrop,
                    prosome_major_predicted, prosome_minor_predicted,
                    lipid_major_predicted, lipid_minor_predicted,
                    lipidPcrop_major_predicted, lipidPcrop_minor_predicted,
                    lipidfullness1, lipidfullness2])

# Create and export the dataframe of output data
df = pd.DataFrame(outdata, 
                  columns = ['directory','filename',
                             'predLipidSac','prob_0','prob_1',
                             'prosome_pixels_predicted', 
                             'lipid_pixels_predicted','lipidpcrop_pixels_predicted',
                             'prosome_major_predicted','prosome_minor_predicted',
                             'lipid_major_predicted','lipid_minor_predicted',
                             'lipidpcrop_major_predicted', 'lipidpcrop_minor_predicted',
                             'lipidfullness1','lipidfullness2'])

df.to_csv(os.path.join(save_dir,'new_predictions_uvp6_20250319.csv'), index=False)

print("Prediction and output done")

In [ ]:
# Prediction with manually annotated *lipid* segmentations
For now, this is applied to the downscaled LOKI images by 1/4 of the resolution. Thus the pixel size would be 92 um instead of 23 um

# Prediction of downscaled LOKI images to 1/4 resolution

In [ ]:
imgs = sorted(list(Path("../downscaled_4/data/segmentations_processed").glob("*.bmp")))
# imgs = imgs[:10]
# print(imgs)

In [ ]:
# Initialize an empty list to store output data metrics
outdata = []

# Image directory
save_dir = "prediction_outputs/model_resnet34_lokid4"

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

for pimg in imgs:
    p_segmentation = pimg
    impath = segmentation2image_path(pimg)
    
    # Predict if lipid sac can be manually segmented
    predLipidSac, pred_idx, probs = modelLipidClassifier.predict(impath)
    prob_0, prob_1 = probs.tolist()

    # Predict prosome
    maskProsome, *_ = modelProsome.predict(impath)
    im_pred_mask_prosome = maskProsome.numpy()
    pixels_predicted_prosome = (im_pred_mask_prosome > 0.5).sum()
    prosome_major_predicted, prosome_minor_predicted = calculate_major_minor_axes(im_pred_mask_prosome)

    # Predict lipid
    maskLipid, *_ = modelLipid.predict(impath)
    im_pred_mask_lipid = maskLipid.numpy()
    pixels_predicted_lipid = (im_pred_mask_lipid > 0.5).sum()
    lipid_major_predicted, lipid_minor_predicted = calculate_major_minor_axes(im_pred_mask_lipid)

    # Crop the prosome out first then apply the lipid model to that
    masked_image = mask_outside_region(impath, im_pred_mask_prosome)
    cv2.imwrite("masked_temp.png", masked_image)
    
    # Predict lipid on the masked image
    maskLipidPcrop, *_ = modelLipidPcrop.predict("masked_temp.png")
    im_pred_mask_lipidPcrop = maskLipidPcrop.numpy()
    pixels_predicted_lipidPcrop = (im_pred_mask_lipidPcrop > 0.5).sum()
    lipidPcrop_major_predicted, lipidPcrop_minor_predicted = calculate_major_minor_axes(im_pred_mask_lipidPcrop)
    
    # Calculate lipid fullness as pixels lipid/pixels prosome
    lipidfullness1 = pixels_predicted_lipid / pixels_predicted_prosome * 100
    lipidfullness2 = pixels_predicted_lipidPcrop / pixels_predicted_prosome * 100

    # TODO Add annotated information


    im = Image.open(impath)

    if p_segmentation is not None:
        im_truth = np.array(Image.open(p_segmentation).convert("L"))
    else:
        im_truth = np.zeros_like(im_pred_mask)
    
    # Calculate metrics
    pixels_annotated = (im_truth > 0).sum()
    lipid_annotated = area2mass(pixels2areaLOKI4(pixels_annotated))
    lipid_predicted = area2mass(pixels2areaLOKI4(pixels_predicted_lipidPcrop))
    IoU, Recall, F1score = calculate_performance_metrics(im_truth, im_pred_mask_lipidPcrop)
    lipid_major_annotated, lipid_minor_annotated = calculate_major_minor_axes(im_pred_mask_lipidPcrop)

    img_with_masks = add_mask2(im, im_truth, im_pred_mask_lipidPcrop)
    # Blue - annotation
    # Green - prediction
    # Purple - overlap
    image_rgb = img_with_masks.convert("RGB")


    out_dir, out_fname = os.path.split(pimg) # Separate the directory from the file name
    filename_without_ext, ext = os.path.splitext(out_fname) # separate the extention
    updated_filename = f"{filename_without_ext}_overlapRGB{ext}"

    # Define the output path
    output_path = os.path.join(save_dir, updated_filename)
    image_rgb.save(output_path)
    
    
    outdata.append([out_dir, out_fname, 
                    predLipidSac,prob_0,prob_1,
                    pixels_predicted_prosome, 
                    pixels_predicted_lipid, pixels_predicted_lipidPcrop,
                    prosome_major_predicted, prosome_minor_predicted,
                    lipid_major_predicted, lipid_minor_predicted,
                    lipidPcrop_major_predicted, lipidPcrop_minor_predicted,
                    lipidfullness1, lipidfullness2,
                    pixels_annotated, lipid_annotated,lipid_predicted,
                    lipid_major_annotated, lipid_minor_annotated,
                    IoU, Recall, F1score])


# Create and export the dataframe of output data
df = pd.DataFrame(outdata, 
                  columns = ['directory','filename',
                             'predLipidSac','prob_0','prob_1',
                             'prosome_pixels_predicted', 
                             'lipid_pixels_predicted','lipidpcrop_pixels_predicted',
                             'prosome_major_predicted','prosome_minor_predicted',
                             'lipid_major_predicted','lipid_minor_predicted',
                             'lipidpcrop_major_predicted', 'lipidpcrop_minor_predicted',
                             'lipidfullness1','lipidfullness2',
                             'lipid_pixels_annotated','lipid_mass_annotated', 'lipid_predicted',
                             'lipid_major_annotated', 'lipid_minor_annotated',
                             'IoU','Recall', 'F1score'])

df.to_csv(os.path.join(save_dir,'new_predictions_LOKId4_20250319.csv'), index=False)

print("Prediction and output done")

In [ ]:
import torch
print(torch.cuda.is_available())  # Should return True if GPU is available